# FFusion 3.5.0 — Version 2 - auto-mode (headless, automatic, no human in the loop)

> Pulls the source image + target video from **DagsHub storage** then runs FF in a headless mode, then auto-saves the output back to DAGS.

Expects:
- target.mp4
- source.jpg

Pulls the source image + target video from **DagsHub** storage, runs FaceFusion **headless** (no UI), then pushes the result back to DagsHub.

Use a **GPU runtime** (Runtime → Change runtime type → T4 GPU, then Disconnect-and-delete + Reconnect).

**Setup once:** add a Colab Secret named `DAGSHUB_USER_TOKEN` (left sidebar → 🔑) holding a valid DagsHub token. Do **not** hardcode the token — the old one was compromised and must be rotated.

In [ ]:
# === CELL 1: VERIFY GPU + CLONE THE FORK ===
!nvidia-smi
%cd /content
!rm -rf facefusion
!git clone https://github.com/alt3rhugo/ff.git facefusion
%cd facefusion

Wed Jun 24 14:33:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   46C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# === CELL 2: INSTALL DEPENDENCIES (latest, via the repo's own installer) ===
!python install.py --onnxruntime cuda --skip-conda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 131.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 130.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.8/252.8 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.6/324.6 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# === CELL 3: DAGSHUB SYSTEM DEPS + CLIENT ===
!curl -s https://rclone.org/install.sh | sudo bash
!apt-get update -y && apt-get install -y fuse3
%pip install -q "dagshub[jupyter]"

Archive:  rclone-current-linux-amd64.zip
   creating: tmp_unzip_dir_for_rclone/rclone-v1.74.3-linux-amd64/
  inflating: tmp_unzip_dir_for_rclone/rclone-v1.74.3-linux-amd64/README.html  [text]  
  inflating: tmp_unzip_dir_for_rclone/rclone-v1.74.3-linux-amd64/README.txt  [text]  
  inflating: tmp_unzip_dir_for_rclone/rclone-v1.74.3-linux-amd64/rclone  [binary]
  inflating: tmp_unzip_dir_for_rclone/rclone-v1.74.3-linux-amd64/rclone.1  [text]  
  inflating: tmp_unzip_dir_for_rclone/rclone-v1.74.3-linux-amd64/git-log.txt  [text]  
Purging old database entries in /usr/share/man...
Processing manual pages under /usr/share/man...
Purging old database entries in /usr/share/man/sv...
Processing manual pages under /usr/share/man/sv...
Purging old database entries in /usr/share/man/de...
Processing manual pages under /usr/share/man/de...
Purging old database entries in /usr/share/man/hu...
Processing manual pages under /usr/share/man/hu...
Purging old database entries in /usr/share/man/ko...
Proc

In [ ]:
# === CELL 4: CONFIG + TOKEN (from Colab Secret, no interactive auth) ===
import os
from google.colab import userdata
from datetime import datetime

timestamp = datetime.now().strftime("%H-%M-%S")

os.environ['DAGSHUB_USER_TOKEN'] = userdata.get('DAGSHUB_USER_TOKEN')

REPO_OWNER = 'zbynja'
REPO_NAME = 'ff'
REPO = f'{REPO_OWNER}/{REPO_NAME}'

# Paths inside DagsHub storage (adjust to match what you upload there).
DAGSHUB_SOURCE = 'source.jpg'         # the face to swap in
DAGSHUB_TARGET = 'target.mp4'         # the video to process
DAGSHUB_OUTPUT = f'output/output_{timestamp}.mp4'  # where the result is written back

LOCAL_SOURCE = '/content/input/source.jpg'
LOCAL_TARGET = '/content/input/target.mp4'
LOCAL_FILENAME = f'output_{timestamp}.mp4'
LOCAL_OUTPUT = f'/content/output/{LOCAL_FILENAME}'

In [ ]:
# === CELL 5: DOWNLOAD SOURCE + TARGET FROM DAGSHUB ===
import shutil
import dagshub
import dagshub.storage

mount_path = dagshub.storage.mount(REPO)
print('Mounted at:', mount_path)
print('Root contents:', os.listdir(mount_path))

os.makedirs('/content/input', exist_ok=True)
os.makedirs('/content/output', exist_ok=True)

shutil.copy(os.path.join(mount_path, DAGSHUB_SOURCE), LOCAL_SOURCE)
shutil.copy(os.path.join(mount_path, DAGSHUB_TARGET), LOCAL_TARGET)
print(f'\u2713 source -> {LOCAL_SOURCE}')
print(f'\u2713 target -> {LOCAL_TARGET}')

Accessing as zbynja

Successfully mounted DagsHub Storage in 'ff' to 'ff/dagshub_storage'.

To unmount, run `dagshub.storage.unmount(repo="zbynja/ff", path="ff/dagshub_storage")`.

Mounted at: ff/dagshub_storage
Root contents: ['---news-reporter-beach-yellow-bikini-p01 ls65c-.mp4', 'source.jpg', 'target.mp4']
✓ source -> /content/input/source.jpg
✓ target -> /content/input/target.mp4


In [ ]:
# === CELL 6: HEADLESS RUN (no UI, no clicks) ===
!python facefusion.py headless-run \
  --execution-providers cuda \
  -s {LOCAL_SOURCE} \
  -t {LOCAL_TARGET} \
  -o {LOCAL_OUTPUT} \
  --processors face_swapper face_enhancer \
  --face-enhancer-model gpen_bfr_512 \
  --face-swapper-model hyperswap_1b_256 \
  --face-swapper-pixel-boost 512x512 \
  --face-enhancer-blend 65 \
  --face-enhancer-weight 0.65 \
  --face-swapper-weight 0.15 \
  --face-mask-blur 0.65 \
  --face-selector-mode one \
  --face-detector-score 0.65 \
  --face-landmarker-score 0.65 \
  --face-detector-angles 0 90 270 \
  --face-mask-types box occlusion

assert os.path.exists(LOCAL_OUTPUT), f'Headless run produced no output at {LOCAL_OUTPUT}'

[FACEFUSION.CORE] processing step 1 of 1
downloading: 100% 8.00/8.00 [00:00<00:00, 58.3B/s, download_providers=['github', 'huggingface'], file_name=nsfw_1.hash]
downloading: 100% 8.00/8.00 [00:00<00:00, 59.1B/s, download_providers=['github', 'huggingface'], file_name=nsfw_2.hash]
downloading: 100% 8.00/8.00 [00:00<00:00, 57.5B/s, download_providers=['github', 'huggingface'], file_name=nsfw_3.hash]
downloading: 100% 76.7M/76.7M [00:00<00:00, 148MB/s, download_providers=['github', 'huggingface'], file_name=nsfw_1.onnx]
downloading: 100% 21.4M/21.4M [00:00<00:00, 74.4MB/s, download_providers=['github', 'huggingface'], file_name=nsfw_2.onnx]
downloading: 100% 342M/342M [00:01<00:00, 239MB/s, download_providers=['github', 'huggingface'], file_name=nsfw_3.onnx]
downloading: 100% 8.00/8.00 [00:00<00:00, 57.1B/s, download_providers=['github', 'huggingface'], file_name=fairface.hash]
downloading: 100% 81.2M/81.2M [00:00<00:00, 225MB/s, download_providers=['github', 'huggingface'], file_name=fai

In [ ]:
# === CELL 7: UPLOAD RESULT TO DAGSHUB STORAGE BUCKET (S3) ===
# Writes the output into the SAME DagsHub Storage Bucket that CELL 5 reads the
# input from -- not the Git/DVC repo. DagsHub buckets are S3-compatible; the
# DagsHub token serves as both the access key and the secret key.
%pip install -q boto3
import boto3

s3 = boto3.client(
    's3',
    endpoint_url=f'https://dagshub.com/api/v1/repo-buckets/s3/{REPO_OWNER}',
    aws_access_key_id=os.environ['DAGSHUB_USER_TOKEN'],
    aws_secret_access_key=os.environ['DAGSHUB_USER_TOKEN'],
)

# Bucket name == repo name; key == the path inside the bucket.
s3.upload_file(LOCAL_OUTPUT, REPO_NAME, DAGSHUB_OUTPUT)
print(f'✓ uploaded -> s3://{REPO_NAME}/{DAGSHUB_OUTPUT} (DagsHub Storage Bucket)')

✓ uploaded -> s3://ff/output/output_14-35-08.mp4 (DagsHub Storage Bucket)


In [ ]:
# Disconnect and delete runtime
from google.colab import runtime
runtime.unassign()